In [3]:
import jax
import jax.numpy as jnp

# Configure JAX before importing anything else
jax.config.update("jax_enable_x64", False)  # Metal GPU requires float32
jax.config.update("jax_platform_name", "metal")  # Enable Metal backend

print("JAX version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default device:", jax.default_backend())

# Test if JAX can actually use the GPU
x = jnp.array([1, 2, 3])
device = jax.devices()[0]  # Get the first available device
x = jax.device_put(x, device)  # Explicitly place array on device
print("Device of test array:", x.device)

JAX version: 0.4.38
Available devices: [METAL(id=0)]
Default device: METAL
Device of test array: METAL:0


In [5]:

from jax import vmap
import numpyro
import numpyro.distributions as dist
from numpyro import sample
import multiprocessing
import os

# Constants
c = 299792.458  # speed of light in km/s

def H(z, H0, Om):
    """Hubble parameter at redshift z"""
    OL = 1 - Om  # Flat universe: Omega_Lambda = 1 - Omega_m
    return H0 * jnp.sqrt(jnp.maximum(Om * (1 + z)**3 + OL, 1e-10))

def luminosity_distance(z, Om, H0):
    """Calculate luminosity distance in Mpc"""
    # Simple rectangle rule integration for comoving distance
    N = 1000  # number of points
    z_array = jnp.linspace(0, z, N)
    dz = z_array[1] - z_array[0] # step size
    
    # Add small epsilon to prevent division by zero
    integrand_values = vmap(lambda z_: c / H(z_, H0, Om))(z_array)
    chi = jnp.sum(integrand_values) * dz
    
    # For flat universe, comoving distance equals proper distance
    return jnp.maximum((1 + z) * chi, 1e-10)  # Ensure positive distance

def distance_modulus(z, Om, H0):
    """Calculate distance modulus"""
    dL = luminosity_distance(z, Om, H0)
    return 5 * jnp.log10(jnp.maximum(dL, 1e-10)) + 25

# Vectorize the distance modulus calculation
distance_modulus_vec = vmap(distance_modulus, in_axes=(0, None, None))

def model(z, mu_obs, mu_err):
    # Priors
    H0 = sample("H0", dist.Uniform(60, 80))     # Uniform prior on H0
    Om = sample("Om", dist.Uniform(0.1, 0.9))   # Uniform prior on Omega_m
    
    # Calculate expected distance modulus
    mu_exp = distance_modulus_vec(z, Om, H0)
    
    # Likelihood (assuming independent measurements)
    sample("obs", dist.Normal(mu_exp, mu_err), obs=mu_obs)

In [7]:
import os

def set_jax_config():
    # Force CPU for better stability with MCMC
    os.environ['JAX_PLATFORM_NAME'] = 'cpu'
    
    # Configure thread count for better performance
    desired_cores = 10  # Adjust based on your CPU
    os.environ["XLA_FLAGS"] = f"--xla_force_host_platform_device_count={desired_cores}"
    os.environ["OPENBLAS_NUM_THREADS"] = str(desired_cores)
    os.environ["MKL_NUM_THREADS"] = str(desired_cores)
    
# Make sure this is called before any other imports
set_jax_config()

import jax
import jax.numpy as jnp
import numpy as np

import numpyro
import numpyro.distributions as dist
from numpyro import sample
import multiprocessing
import pandas as pd
import matplotlib.pyplot as plt

def load_data():
    df = pd.read_csv('/Users/yhra/Documents/Master/Semester_3/BATIP/Supernova_project/Data/Pantheon+SH0ES.dat', sep='\s+', header=0)
    return df

if __name__ == "__main__":
    df = load_data()
    z = df['zHD'].values
    mu = df['MU_SH0ES'].values
    mu_err = df['MU_SH0ES_ERR_DIAG'].values

    rng_key = jax.random.PRNGKey(42)
    
    # Add initialization strategy and configure NUTS
    init_strategy = numpyro.infer.init_to_median()
    kernel = numpyro.infer.NUTS(
        model,
        target_accept_prob=0.8,  # Slightly more conservative
        max_tree_depth=4,        # Allow for more complex trajectories    )
    )

    mcmc = numpyro.infer.MCMC(
        kernel, 
        num_warmup=500, 
        num_samples=2000
    )
    # Initialize the model with median values before running
    mcmc.run(rng_key, z, mu, mu_err)
    samples = mcmc.get_samples()

    # from this distribution calculate Omega_lambda
    Omega_lambda = 1 - samples['Om'] 
    
    # Create a figure with four subplots
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))
    
    # Plot H0 distribution
    ax1.hist(samples['H0'], bins=50, color='skyblue', edgecolor='black', density=True)
    ax1.set_title('H₀ Distribution')
    ax1.set_xlabel('H₀ [km/s/Mpc]')
    ax1.set_ylabel('Frequency')
    
    # Plot Omega_m distribution
    ax2.hist(samples['Om'], bins=50, color='lightgreen', edgecolor='black', density=True)
    ax2.set_title('Ωₘ Distribution')
    ax2.set_xlabel('Ωₘ')
    ax2.set_ylabel('Frequency')
    
    # Plot Omega_lambda distribution
    ax3.hist(Omega_lambda, bins=50, color='salmon', edgecolor='black', density=True)
    ax3.set_title('Ωₗ Distribution')
    ax3.set_xlabel('Ωₗ')
    ax3.set_ylabel('Frequency')

    # Adjust layout and display
    plt.tight_layout()
    plt.show()

  0%|          | 0/2500 [00:00<?, ?it/s]


XlaRuntimeError: INTERNAL: Unable to serialize MPS module